<a href="https://colab.research.google.com/github/richard-alcala-code/AIOMovie/blob/develop/notebooks/AIO_Movie_Core_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. MovieLens Configuration and Loading

In [42]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# 2. Define the file paths
base_path = '/content/drive/MyDrive/Master2026/NeuralNetworksAndDeepLearning/AIOMovie/DataSets/MovieLens/'
path_movies = base_path + 'movies.csv'
path_links = base_path + 'links.csv'
path_ratings = base_path + 'ratings.csv'

try:
    # 3. Load all necessary datasets
    movies = pd.read_csv(path_movies)
    links = pd.read_csv(path_links)
    ratings = pd.read_csv(path_ratings)

    # 4. Merge them to create a complete training dataset
    # We merge ratings with links to get both 'userId' and 'tmdbId' in one table
    movie_data = pd.merge(ratings, links, on='movieId')

    # Merge with titles
    movie_data = pd.merge(movie_data, movies[['movieId', 'title']], on='movieId')

    print(f"Dataset loaded successfully with {movie_data.shape} rating records.")
    print(f"Columns now available: {movie_data.columns.tolist()}")

except FileNotFoundError:
    print("Error: Ensure movies.csv, links.csv, and ratings.csv are in your Drive.")

Mounted at /content/drive
Dataset loaded successfully with (100836, 7) rating records.
Columns now available: ['userId', 'movieId', 'rating', 'timestamp', 'imdbId', 'tmdbId', 'title']


## 2. Metadata Integration with TMDB Token

In [43]:
import requests
from google.colab import userdata

TMDB_TOKEN = userdata.get('TMDB_TOKEN')
BASE_URL = "https://api.themoviedb.org/3/movie/"

def get_movie_metadata(tmdb_id):
    # Ensure tmdb_id is not NaN and can be converted to int
    if pd.isna(tmdb_id):
        return None

    headers = {
        "Authorization": f"Bearer {TMDB_TOKEN}",
        "Content-Type": "application/json;charset=utf-8"
    }
    try:
        # Query TMDB using the ID obtained from MovieLens
        response = requests.get(f"{BASE_URL}{int(tmdb_id)}", headers=headers)
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        data = response.json()
        return {
            "overview": data.get("overview"),
            "poster_path": f"https://image.tmdb.org/t/p/w500{data.get('poster_path')}"
        }
    except requests.exceptions.RequestException as e:
        print(f"API request failed for TMDB ID {tmdb_id}: {e}")
        return None
    except ValueError as e:
        print(f"Error converting TMDB ID {tmdb_id} to int: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred for TMDB ID {tmdb_id}: {e}")
        return None

# Test example with a movie
# Select the first valid tmdbId for testing
first_tmdb_id = movie_data['tmdbId'].dropna().iloc[0]
test_metadata = get_movie_metadata(first_tmdb_id)

# Check if test_metadata is not None before accessing its elements
if test_metadata:
    print(f"Synopsis: {test_metadata['overview'][:100]}...")
else:
    print(f"Could not retrieve metadata for TMDB ID: {first_tmdb_id}")

Synopsis: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto ...


## 3. Deep Learning Model Architecture (Embeddings)

In [44]:
import tensorflow as tf
from tensorflow.keras import layers

def build_recommender_model(num_users, num_movies, embedding_size=50):
    # Inputs for User and Movie IDs
    user_input = layers.Input(shape=(1,), name="user_input")
    movie_input = layers.Input(shape=(1,), name="movie_input")

    # User Embeddings
    user_embedding = layers.Embedding(num_users, embedding_size, name="user_embedding")(user_input)
    user_vec = layers.Flatten()(user_embedding)

    # Movie Embeddings
    movie_embedding = layers.Embedding(num_movies, embedding_size, name="movie_embedding")(movie_input)
    movie_vec = layers.Flatten()(movie_embedding)

    # Dot product to find similarity between User and Movie
    dot_product = layers.Dot(axes=1)([user_vec, movie_vec])

    model = tf.keras.Model(inputs=[user_input, movie_input], outputs=dot_product)
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

print("Initial semantic recommendation model architecture initialized.")

Initial semantic recommendation model architecture initialized.


## 4. Training and Generating Top 10 Recommendations

In [45]:
import numpy as np

# --- Data Preparation ---
# Encode IDs to be continuous (0 to N)
user_ids = movie_data["userId"].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
# movie_ids = movie_data["tmdbId"].unique().tolist()
movie_ids = sorted(movie_data["tmdbId"].unique().tolist())
movie2movie_encoded = {x: i for i, x in enumerate(movie_ids)}

movie_data["user"] = movie_data["userId"].map(user2user_encoded)
movie_data["movie"] = movie_data["tmdbId"].map(movie2movie_encoded)

# --- Model Training ---
model = build_recommender_model(len(user_ids), len(movie_ids))
model.fit(
    x=[movie_data["user"], movie_data["movie"]],
    y=movie_data["rating"],
    batch_size=64,
    epochs=5
)

# --- Generate Top 10 Recommendations for a Test User ---
# Select a single test user, for example, the first user in the list
test_user_id = user_ids[0] # Changed from test_user_id = user_ids
movies_watched = movie_data[movie_data.userId == test_user_id].tmdbId.values
movies_not_watched = [m for m in movie_ids if m not in movies_watched]
movies_not_watched_encoded = [[movie2movie_encoded.get(x)] for x in movies_not_watched]

user_encoder = user2user_encoded.get(test_user_id)
user_movie_array = np.hstack(
    ([[user_encoder]] * len(movies_not_watched), movies_not_watched_encoded)
)

predictions = model.predict([user_movie_array[:,0], user_movie_array[:,1]]).flatten()
top_indices = predictions.argsort()[-10:][::-1]
recommended_ids = [movies_not_watched[i] for i in top_indices]

print(f"Top 10 Recommendations for User {test_user_id}: {recommended_ids}")

Epoch 1/5
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 8.8559 
Epoch 2/5
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.7883
Epoch 3/5
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.0813
Epoch 4/5
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.8663
Epoch 5/5
1576/1576 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.7512
297/297 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Top 10 Recommendations for User 1: [359940.0, 37916.0, 3114.0, 32636.0, 2117.0, 13321.0, 49020.0, 2440.0, 14886.0, 1480.0]


## 5. Save the model (.keras)

In [46]:
# Save the model in the native Keras format
model.save('aiomovie_model.keras')
print("Model saved as 'aiomovie_model.keras'.")

Model saved as 'aiomovie_model.keras'.


## 6. Generate Movie-Based Recommendations (Using the re-generated model)

In [47]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# --- Data Loading
# Assuming Google Drive is mounted. For robustness, redefine base_path and load dataframes.
base_path = '/content/drive/MyDrive/Master2026/NeuralNetworksAndDeepLearning/AIOMovie/DataSets/MovieLens/'
path_movies = base_path + 'movies.csv'
path_links = base_path + 'links.csv'
path_ratings = base_path + 'ratings.csv'

# Initialize these to None or empty to prevent NameError in case of FileNotFoundError
movies = None
links = None
ratings = None
movie_data = None
tmdb_id_to_title_map = {} # Initialize as empty dict
data_loaded_successfully = False # Flag to track data loading status

try:
    movies = pd.read_csv(path_movies)
    links = pd.read_csv(path_links)
    ratings = pd.read_csv(path_ratings)

    # Re-create movie_data for this cell's scope
    movie_data = pd.merge(ratings, links, on='movieId')
    movie_data = pd.merge(movie_data, movies[['movieId', 'title']], on='movieId')

    # Create a mapping from tmdbId to title for efficient title retrieval
    # This block now correctly resides AFTER movies and links are loaded within the try block.
    tmdb_id_to_title_map_df = pd.merge(movies[['movieId', 'title']], links[['movieId', 'tmdbId']], on='movieId', how='inner')
    tmdb_id_to_title_map = tmdb_id_to_title_map_df.drop_duplicates(subset=['tmdbId']).set_index('tmdbId')['title'].to_dict()

    data_loaded_successfully = True # Set flag to True if data loading is successful

except FileNotFoundError:
    print("Error: Ensure movies.csv, links.csv, and ratings.csv are in your Drive.")
    data_loaded_successfully = False # Set flag to False if data loading fails

# --- End Data Loading ---

# Only define and use functions if data was loaded successfully
if data_loaded_successfully:
    def get_movie_info_by_title(title_substring, movies_df, movie_data_df):
        """
        Searches for a movie by a substring of its title (case-insensitive)
        and returns its tmdbId and the exact found title.
        Uses movies_df for initial title search and movie_data_df to link to tmdbId.
        """
        # Search in the original 'movies' DataFrame for a clean title match
        matching_movies_in_movies = movies_df[movies_df['title'].str.contains(title_substring, case=False, na=False, regex=False)]

        if not matching_movies_in_movies.empty:
            # Get the movieId and exact title from the first match in 'movies'
            matched_movie_id = matching_movies_in_movies.iloc[0]['movieId']
            exact_found_title = matching_movies_in_movies.iloc[0]['title']

            # Now, find the corresponding tmdbId from the 'movie_data' DataFrame
            # which contains the movieId-tmdbId mapping
            tmdb_id_row = movie_data_df[movie_data_df['movieId'] == matched_movie_id]
            if not tmdb_id_row.empty:
                tmdb_id = tmdb_id_row.iloc[0]['tmdbId']
                return tmdb_id, exact_found_title
        return None, None

    def recommend_movies_based_on_movie(input_movie_title, model, movie2movie_encoded, movies_df, tmdb_id_to_title_map, top_n=10):
        """
        Generates movie recommendations based on a given movie title using movie embeddings.
        """
        target_tmdb_id, found_title = get_movie_info_by_title(input_movie_title, movies_df, movie_data) # Pass movie_data here
        if target_tmdb_id is None:
            print(f"Movie '{input_movie_title}' not found in the dataset.")
            return []

        target_movie_encoded = movie2movie_encoded.get(target_tmdb_id)
        if target_movie_encoded is None:
            print(f"Encoded ID for '{found_title}' (TMDB ID: {target_tmdb_id}) not found. This movie might not be in the training data.")
            return []

        # Get the movie embedding layer from the trained model
        movie_embedding_layer = model.get_layer('movie_embedding')
        movie_embeddings = movie_embedding_layer.get_weights()[0] # Shape: (num_movies, embedding_size)

        # Get the embedding of the target movie
        target_movie_embedding = movie_embeddings[target_movie_encoded]

        # Calculate cosine similarity with all other movie embeddings
        similarities = cosine_similarity(target_movie_embedding.reshape(1, -1), movie_embeddings).flatten()

        # Get the indices of the most similar movies (excluding the movie itself)
        # argsort returns indices that would sort an array; [::-1] reverses for descending order
        # [1:top_n+1] slices to get top N, excluding the first one (which is the movie itself)
        top_similar_encoded_indices = similarities.argsort()[::-1][1:top_n+1]

        recommended_titles = []
        # Create a reverse mapping from encoded_id to tmdb_id
        movie_encoded_to_tmdb = {v: k for k, v in movie2movie_encoded.items()}

        for encoded_idx in top_similar_encoded_indices:
            recommended_tmdb_id = movie_encoded_to_tmdb.get(encoded_idx)
            if recommended_tmdb_id is not None:
                # Use the pre-created map for titles
                title = tmdb_id_to_title_map.get(recommended_tmdb_id)
                if title:
                    recommended_titles.append(title)

        return recommended_titles

    # --- Example Usage ---
    movie_to_search = "Toy Story" # Example movie title
    recommended_movies = recommend_movies_based_on_movie(movie_to_search, model, movie2movie_encoded, movies, tmdb_id_to_title_map, top_n=10)

    if recommended_movies:
        print(f"\nTop 10 recommendations for '{movie_to_search}':")
        for i, title in enumerate(recommended_movies):
            print(f"{i+1}. {title}")
    else:
        print(f"\nNo recommendations could be generated for '{movie_to_search}'.")

    movie_to_search_2 = "Matrix" # Example movie title
    recommended_movies_2 = recommend_movies_based_on_movie(movie_to_search_2, model, movie2movie_encoded, movies, tmdb_id_to_title_map, top_n=5)
    if recommended_movies_2:
        print(f"\nTop 5 recommendations for '{movie_to_search_2}':")
        for i, title in enumerate(recommended_movies_2):
            print(f"{i+1}. {title}")
    else:
        print(f"\nNo recommendations could be generated for '{movie_to_search_2}'.")



Top 10 recommendations for 'Toy Story':
1. Fugitive, The (1993)
2. Mission: Impossible (1996)
3. Jurassic Park (1993)
4. Rock, The (1996)
5. Dances with Wolves (1990)
6. Braveheart (1995)
7. Lion King, The (1994)
8. Die Hard (1988)
9. Star Wars: Episode IV - A New Hope (1977)
10. Beauty and the Beast (1991)

Top 5 recommendations for 'Matrix':
1. Saving Private Ryan (1998)
2. Back to the Future (1985)
3. Shawshank Redemption, The (1994)
4. Schindler's List (1993)
5. Indiana Jones and the Temple of Doom (1984)


6. Generate Movie-Based Recommendations (Based on existing model)

## 7. Generate Movie-Based Recommendations (Using an alternative model from Drive)

In [49]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive
import os

# Mount Google Drive if not already mounted
drive.mount("/content/drive", force_remount=True)

# Define the path to the alternative model
alternative_model_path = '/content/drive/MyDrive/Master2026/NeuralNetworksAndDeepLearning/AIOMovie/Model/aiomovie_model_v3.keras'

# --- Load the Alternative Model ---
alternative_model = None
try:
    if os.path.exists(alternative_model_path):
        alternative_model = tf.keras.models.load_model(alternative_model_path)
        print(f"Alternative model loaded successfully from '{alternative_model_path}'.")
    else:
        raise FileNotFoundError(f"Alternative model not found at: {alternative_model_path}")
except Exception as e:
    print(f"Error loading alternative model: {e}")

def get_movie_info_by_title(title_substring, movies_df, movie_data_df):
    """
    Searches for a movie by a substring of its title (case-insensitive)
    and returns its tmdbId and the exact found title.
    Uses movies_df for initial title search and movie_data_df to link to tmdbId.
    """
    matching_movies_in_movies = movies_df[movies_df['title'].str.contains(title_substring, case=False, na=False, regex=False)]

    if not matching_movies_in_movies.empty:
        matched_movie_id = matching_movies_in_movies.iloc[0]['movieId']
        exact_found_title = matching_movies_in_movies.iloc[0]['title']

        tmdb_id_row = movie_data_df[movie_data_df['movieId'] == matched_movie_id]
        if not tmdb_id_row.empty and pd.notna(tmdb_id_row.iloc[0]['tmdbId']):
            tmdb_id = tmdb_id_row.iloc[0]['tmdbId']
            return tmdb_id, exact_found_title
    return None, None

def recommend_movies_based_on_movie(input_movie_title, model_to_use, movie2movie_encoded, movies_df, tmdb_id_to_title_map, top_n=10):
    """
    Generates movie recommendations based on a given movie title using movie embeddings.
    """
    if model_to_use is None:
        return [] # Return empty list if model wasn't loaded

    target_tmdb_id, found_title = get_movie_info_by_title(input_movie_title, movies_df, movie_data)
    if target_tmdb_id is None:
        print(f"Movie '{input_movie_title}' not found in the dataset.")
        return []

    target_movie_encoded = movie2movie_encoded.get(target_tmdb_id)
    if target_movie_encoded is None:
        print(f"Encoded ID for '{found_title}' (TMDB ID: {target_tmdb_id}) not found. This movie might not be in the training data.")
        return []

    movie_embedding_layer = model_to_use.get_layer('movie_embedding')
    movie_embeddings = movie_embedding_layer.get_weights()[0]

    if target_movie_encoded >= len(movie_embeddings):
        print(f"Error: Encoded ID {target_movie_encoded} for '{found_title}' is out of bounds for movie embeddings.")
        return []

    target_movie_embedding = movie_embeddings[target_movie_encoded]

    similarities = cosine_similarity(target_movie_embedding.reshape(1, -1), movie_embeddings).flatten()

    top_similar_encoded_indices = similarities.argsort()[::-1][1:top_n+1]

    recommended_titles = []
    movie_encoded_to_tmdb = {v: k for k, v in movie2movie_encoded.items()}

    for encoded_idx in top_similar_encoded_indices:
        recommended_tmdb_id = movie_encoded_to_tmdb.get(encoded_idx)
        if recommended_tmdb_id is not None:
            title = tmdb_id_to_title_map.get(recommended_tmdb_id)
            if title:
                recommended_titles.append(title)

    return recommended_titles

# --- Example Usage with the Alternative Model ---

if alternative_model:
    print(f"\nGenerating recommendations using alternative model: {alternative_model_path}")
    movie_to_search_alt = "Toy Story" # Example movie title
    recommended_movies_alt = recommend_movies_based_on_movie(movie_to_search_alt, alternative_model, movie2movie_encoded, movies, tmdb_id_to_title_map, top_n=10)

    if recommended_movies_alt:
        print(f"\nTop 10 recommendations for '{movie_to_search_alt}' (using alternative model):")
        for i, title in enumerate(recommended_movies_alt):
            print(f"{i+1}. {title}")
    else:
        print(f"\nNo recommendations could be generated for '{movie_to_search_alt}' using the alternative model.")

    movie_to_search_alt_2 = "Matrix" # Example movie title
    recommended_movies_alt_2 = recommend_movies_based_on_movie(movie_to_search_alt_2, alternative_model, movie2movie_encoded, movies, tmdb_id_to_title_map, top_n=5)
    if recommended_movies_alt_2:
        print(f"\nTop 5 recommendations for '{movie_to_search_alt_2}' (using alternative model):")
        for i, title in enumerate(recommended_movies_alt_2):
            print(f"{i+1}. {title}")
    else:
        print(f"\nNo recommendations could be generated for '{movie_to_search_alt_2}' using the alternative model.")
else:
    print("Cannot generate recommendations: Alternative model was not loaded successfully.")

Mounted at /content/drive
Alternative model loaded successfully from '/content/drive/MyDrive/Master2026/NeuralNetworksAndDeepLearning/AIOMovie/Model/aiomovie_model_v3.keras'.

Generating recommendations using alternative model: /content/drive/MyDrive/Master2026/NeuralNetworksAndDeepLearning/AIOMovie/Model/aiomovie_model_v3.keras

Top 10 recommendations for 'Toy Story' (using alternative model):
1. Fugitive, The (1993)
2. Dances with Wolves (1990)
3. Beauty and the Beast (1991)
4. Philadelphia (1993)
5. Toy Story 2 (1999)
6. Birdcage, The (1996)
7. Dave (1993)
8. Aladdin (1992)
9. Shawshank Redemption, The (1994)
10. Forrest Gump (1994)

Top 5 recommendations for 'Matrix' (using alternative model):
1. Saving Private Ryan (1998)
2. Star Wars: Episode VI - Return of the Jedi (1983)
3. Star Wars: Episode IV - A New Hope (1977)
4. Die Hard (1988)
5. Léon: The Professional (a.k.a. The Professional) (Léon) (1994)
